# 🧭 **General Workflow for a Regression Task (Template Notebook)**

This notebook is the **template workflow** we use throughout the course for regression problems with `scikit-learn`. All regression labs in this repository follow roughly the same structure:

1. **Import Libraries**
2. **Load the Data & describe the Dataset/Task**
3. **Exploratory Data Analysis (EDA)**
4. **Preprocess the Data** — *split first, then standardize*
5. **Train the Model**
6. **Evaluate the Model** (compare with a baseline)
7. **Compare Methods**

> ⚠️ **Golden rule of preprocessing:** Standardization (or any scaling/encoding that is fit from data) must be performed **after** the train/test split. Fitting the scaler on the full dataset leaks information from the test fold into training through the mean and standard deviation.

# 📈 **ML Regression for Advertising Sales**

---
In this lab, you'll learn how to preprocess tabular data and solve a regression problem using **three equivalent techniques**:

1. **Normal Equation** (closed-form solution with matrix inversion)
2. **QR Decomposition** (numerically stable factorisation)
3. **Scikit-Learn `LinearRegression`** (the library way)

All three produce the same least-squares coefficients. The goal is to understand **why** they are equivalent and **when** you might prefer one over another.

---

# 📊 **Dataset & Task**

**Dataset:** [Advertising dataset](https://www.kaggle.com/datasets/bumba5341/advertisingcsv) — 200 observations of advertising budgets (in thousands of dollars) spent across three channels for a single product, with the corresponding sales (in thousands of units).

| Column | Description | Type |
|--------|-------------|------|
| `TV` | Advertising budget on TV | numeric feature |
| `Radio` | Advertising budget on Radio | numeric feature |
| `Newspaper` | Advertising budget on Newspaper | numeric feature |
| `Sales` | Product sales | **target** (continuous) |

**Task:** Predict the continuous target `Sales` from the three advertising budgets. This is a **regression** problem, so we will use models such as **Linear Regression**, **Ridge**, and **LASSO**, and evaluate with **MSE / RMSE / R²**.


# 1️⃣ Import Libraries


> First install the extra dependencies (only needed the first time in a fresh environment).

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

# 2️⃣ Read the Data


In [ ]:
# Download latest version
path = kagglehub.dataset_download("bumba5341/advertisingcsv")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Advertising.csv")

df = pd.read_csv(csv_path)
df = df.drop(columns="Unnamed: 0", axis=1)
df.head()

In [ ]:
df.info()

# 3️⃣ Exploratory Data Analysis (EDA)

**Rule of thumb checklist:**

| Question | If YES | If NO |
|----------|--------|-------|
|  **Is the target skewed?** | Consider log transform | Proceed |
|  **Missing values?** | Impute or drop | Proceed |
|  **Categorical columns?** | Encode them | Proceed |
|  **Duplicates?** | Drop them | Proceed |
|  **Different scales?** | Standardize  | Proceed |


In [ ]:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Sales")

> **Rule of thumb:** For regression, if the target is heavily skewed, we might apply a log transform. However, for this lab we'll work with the raw values.


In [ ]:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

> We don't have categorical columns. If we did, we would need to encode them (convert them into numbers) for our models.

In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

> **Rule of thumb:** If duplicates exist, drop them with `df.drop_duplicates(inplace=True)`


In [ ]:
# 5. Do we have different scales in the data?
df.describe()

> **Rule of thumb:** If features have vastly different ranges, then **scale them**. Scaling won't hurt, and it's recommended regardless.
>
> Tree models (RF, LightGBM, CatBoost) don't need scaling, but Linear Regression, Ridge, LASSO, SVM do!
>
> ⚠️ **We do NOT scale here.** Scaling must be fit on the **training set only** and then applied to the test set. We will therefore scale **inside** the cross-validation loop (after the split) to avoid data leakage.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# We only IMPORT the scaler here. We will fit it on X_train inside the K-Fold loop.
# ❌ DO NOT DO THIS (data leakage):
#     scaler = MinMaxScaler().fit(df[features])
#     df[features] = scaler.transform(df[features])
#
# ✅ Instead: split → fit scaler on X_train → transform X_test (see below).


# 4️⃣ Training our Regression Models

> We need to split our data into **X** (features) and **y** (target).


In [ ]:
X = df.drop("Sales", axis=1).astype(float)
y = df['Sales'].astype(float)

---

# Part A: Normal Equation (Closed-Form Solution)

### **Linear Regression Model**

Linear regression predicts a continuous target value by finding the best linear relationship between features and target:

$$\hat{y} = X \cdot \theta$$

Where:
- $X$ = feature matrix (with bias term)
- $\theta$ = weight vector (parameters to learn)
- $\hat{y}$ = predicted values

---

### **Mean Squared Error Loss (MSE)**

We use **Mean Squared Error** to measure how well our predictions match the true values:

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

Where:
- $m$ = number of samples
- $y_i$ = true value
- $\hat{y}_i$ = predicted value

> **Intuition:** If we predict the exact value, MSE is 0. The bigger the difference, the higher the penalty (and it's squared, so large errors are penalized heavily)!

---

### **Normal Equation**

The closed-form solution sets the gradient of the loss to zero and solves directly:

$$\theta = (X^T X)^{-1} X^T y$$

This gives us the optimal weights in **one shot** — no iterative optimisation needed.

> ⚠️ **Caveat:** Computing $(X^T X)^{-1}$ can be numerically unstable when features are correlated or the matrix is ill-conditioned. That's where QR decomposition helps (see Part B).

In [ ]:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:
def normal_equation(X, y):
   # Add bias term (intercept)
    m = X.shape[0]
    X_b = np.c_[np.ones((m, 1)), X]

    # Closed-form solution using pseudoinverse
    theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    return theta

**Training Linear Regression** using K-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Storage for linear regression results for each fold
train_mse_list = []
test_mse_list = []

train_rmse_list = []
test_rmse_list = []

train_r2_list = []
test_r2_list = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # 1) SPLIT first
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2) SCALE after split: fit on train, transform both train and test
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 3) Train (closed form)
    theta = normal_equation(X_train_scaled, y_train.values)

    # Add bias column
    X_train_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
    X_test_b  = np.c_[np.ones((X_test_scaled.shape[0], 1)),  X_test_scaled]

    # Predictions
    y_train_pred = X_train_b @ theta
    y_test_pred  = X_test_b @ theta

    # Metrics
    train_mse = sklearn_mse(y_train, y_train_pred)
    test_mse  = sklearn_mse(y_test, y_test_pred)

    train_rmse = np.sqrt(train_mse)
    test_rmse  = np.sqrt(test_mse)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2  = r2_score(y_test, y_test_pred)

    # Store
    train_mse_list.append(train_mse)
    test_mse_list.append(test_mse)

    train_rmse_list.append(train_rmse)
    test_rmse_list.append(test_rmse)

    train_r2_list.append(train_r2)
    test_r2_list.append(test_r2)

    print(f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")
    print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f}")


**Linear Regression Loss Curve**

> use `np.mean(..)` to get the average across all folds.


In [ ]:
print("Linear Regression Results")

print(f"  Average Train MSE: {np.mean(train_mse_list):.4f}")
print(f"  Average Test  MSE: {np.mean(test_mse_list):.4f}")

print(f"  Average Train RMSE: {np.mean(train_rmse_list):.4f}")
print(f"  Average Test  RMSE: {np.mean(test_rmse_list):.4f}")

print(f"  Average Train R2: {np.mean(train_r2_list):.4f}")
print(f"  Average Test  R2: {np.mean(test_r2_list):.4f}")

---

# Part B: QR Decomposition

**QR decomposition** is a numerically more stable way to solve the same least-squares problem.

Instead of computing $(X^T X)^{-1}$ directly, we factorise:

$$X = Q R$$

where $Q$ is orthogonal ($Q^T Q = I$) and $R$ is upper triangular. Then:

$$\theta = R^{-1} Q^T y$$

Because $R$ is triangular, solving $R\theta = Q^T y$ is fast (back-substitution) and avoids the numerical issues of inverting $X^T X$.

> **When to prefer QR?** Whenever the feature matrix may be ill-conditioned or you care about floating-point precision (which is almost always in practice).

In [ ]:
def qr_solve(X, y):
    """Solve least-squares via QR decomposition (with bias column)."""
    m = X.shape[0]
    X_b = np.c_[np.ones((m, 1)), X]  # Add bias term
    Q, R = np.linalg.qr(X_b)
    theta = np.linalg.solve(R, Q.T @ y)
    return theta

**Training with QR Decomposition** using K-Fold Cross-Validation

In [ ]:
# Storage for QR results
qr_train_mse_list, qr_test_mse_list = [], []
qr_train_rmse_list, qr_test_rmse_list = [], []
qr_train_r2_list, qr_test_r2_list = [], []

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # 1) SPLIT first
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2) SCALE after split: fit on train, transform both
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 3) Train (QR decomposition)
    theta = qr_solve(X_train_scaled, y_train.values)

    # Add bias column for predictions
    X_train_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
    X_test_b  = np.c_[np.ones((X_test_scaled.shape[0], 1)),  X_test_scaled]

    # Predictions
    y_train_pred = X_train_b @ theta
    y_test_pred  = X_test_b @ theta

    # Metrics
    train_mse = sklearn_mse(y_train, y_train_pred)
    test_mse  = sklearn_mse(y_test, y_test_pred)
    train_rmse, test_rmse = np.sqrt(train_mse), np.sqrt(test_mse)
    train_r2, test_r2 = r2_score(y_train, y_train_pred), r2_score(y_test, y_test_pred)

    qr_train_mse_list.append(train_mse);  qr_test_mse_list.append(test_mse)
    qr_train_rmse_list.append(train_rmse); qr_test_rmse_list.append(test_rmse)
    qr_train_r2_list.append(train_r2);    qr_test_r2_list.append(test_r2)

    print(f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")
    print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f}")

In [ ]:
print("QR Decomposition Results")

print(f"  Average Train MSE: {np.mean(qr_train_mse_list):.4f}")
print(f"  Average Test  MSE: {np.mean(qr_test_mse_list):.4f}")

print(f"  Average Train RMSE: {np.mean(qr_train_rmse_list):.4f}")
print(f"  Average Test  RMSE: {np.mean(qr_test_rmse_list):.4f}")

print(f"  Average Train R2: {np.mean(qr_train_r2_list):.4f}")
print(f"  Average Test  R2: {np.mean(qr_test_r2_list):.4f}")

---

# Part C: Scikit-Learn `LinearRegression`

Scikit-learn's `LinearRegression` wraps the same mathematics (it uses a least-squares solver internally) behind a clean **Instantiate → Fit → Predict** API.

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()   # instantiate
model.fit(X_train, y_train)  # fit
y_pred = model.predict(X_test)  # predict
```

> Under the hood, `LinearRegression` uses a similar decomposition. The advantage is **convenience**: no manual bias column, no manual matrix algebra.

In [ ]:
from sklearn.linear_model import LinearRegression

# Storage for sklearn results
sk_train_mse_list, sk_test_mse_list = [], []
sk_train_rmse_list, sk_test_rmse_list = [], []
sk_train_r2_list, sk_test_r2_list = [], []

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # 1) SPLIT first
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2) SCALE after split
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 3) Train (scikit-learn)
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred  = model.predict(X_test_scaled)

    # Metrics
    train_mse = sklearn_mse(y_train, y_train_pred)
    test_mse  = sklearn_mse(y_test, y_test_pred)
    train_rmse, test_rmse = np.sqrt(train_mse), np.sqrt(test_mse)
    train_r2, test_r2 = r2_score(y_train, y_train_pred), r2_score(y_test, y_test_pred)

    sk_train_mse_list.append(train_mse);  sk_test_mse_list.append(test_mse)
    sk_train_rmse_list.append(train_rmse); sk_test_rmse_list.append(test_rmse)
    sk_train_r2_list.append(train_r2);    sk_test_r2_list.append(test_r2)

    print(f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")
    print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f}")

In [ ]:
print("Scikit-Learn LinearRegression Results")

print(f"  Average Train MSE: {np.mean(sk_train_mse_list):.4f}")
print(f"  Average Test  MSE: {np.mean(sk_test_mse_list):.4f}")

print(f"  Average Train RMSE: {np.mean(sk_train_rmse_list):.4f}")
print(f"  Average Test  RMSE: {np.mean(sk_test_rmse_list):.4f}")

print(f"  Average Train R2: {np.mean(sk_train_r2_list):.4f}")
print(f"  Average Test  R2: {np.mean(sk_test_r2_list):.4f}")

# 5️⃣ Results Comparison

> All three methods solve the **same** ordinary least-squares problem, so their results should be **identical** (up to floating-point precision). The table below confirms this.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": ["Normal Equation", "QR Decomposition", "Sklearn LinearRegression"],
    "Avg Train MSE": [
        np.mean(train_mse_list),
        np.mean(qr_train_mse_list),
        np.mean(sk_train_mse_list),
    ],
    "Avg Test MSE": [
        np.mean(test_mse_list),
        np.mean(qr_test_mse_list),
        np.mean(sk_test_mse_list),
    ],
    "Avg Test RMSE": [
        np.mean(test_rmse_list),
        np.mean(qr_test_rmse_list),
        np.mean(sk_test_rmse_list),
    ],
    "Avg Test R²": [
        np.mean(test_r2_list),
        np.mean(qr_test_r2_list),
        np.mean(sk_test_r2_list),
    ],
})

comparison = comparison.set_index("Method")
comparison.style.format("{:.4f}").set_caption("Comparison of Three Least-Squares Solvers")

### Key Takeaways

| Method | Pros | Cons |
|--------|------|------|
| **Normal Equation** | Simple formula, easy to understand | Numerically unstable for ill-conditioned $X^T X$; requires explicit bias column |
| **QR Decomposition** | Numerically stable; avoids forming $X^T X$ | Slightly more code; requires explicit bias column |
| **Sklearn `LinearRegression`** | Clean API; handles bias automatically; production-ready | Less educational — hides the math |

> **Bottom line:** Use `LinearRegression` in practice. Understand the Normal Equation and QR for the theory behind it.

---

## Are our models good?

>Is RMSE = 1.00 good? How to know if a score is good or not?
>
>We need something to compare against. We need a Baseline.
>
>If our models are better than the baseline score, then they are good.

**The simplest baseline we can consider is the mean of the target**.

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mse = sklearn_mse(y, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y, baseline_pred)

print(f"Baseline MSE (using mean target): {baseline_mse:.4f}")
print(f"Baseline RMSE (using mean target): {baseline_rmse:.4f}")
print(f"Baseline R2 (using mean target): {baseline_r2:.4f}")

> **Our models are way better than this baseline because our MSE is lower than the baseline MSE (variance).**

# 🎯 **Exercises**

The exercises below are small pieces of the workflow you just saw. Replace every `None` or `?` with the correct code. The goal is to help you *internalise* the regression workflow: load → split → scale → train → evaluate.

### Exercise 1 — Split the data

Complete the call to `train_test_split` so that **20% of the samples** go to the test set, and the split is **reproducible** (use `random_state=42`).

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: fill in the ? so that 20% goes to test and the split is reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=?,        # ← ?
    random_state=?      # ← ?
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)

### Exercise 2 — Scale **after** the split (no leakage!)

Complete the next cell so that the scaler is **fit on the training set only**, and then **applied** to the test set.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# TODO: learn the mean/std from X_train AND scale it in one call.
X_train_scaled = None   # ← hint: scaler.fit_transform(...)

# TODO: use the SAME scaler (already fit on train) to transform X_test.
X_test_scaled  = None   # ← hint: scaler.transform(...)

print("mean(X_train_scaled) ≈ 0 ?", X_train_scaled.mean(axis=0).round(3))
print("std (X_train_scaled) ≈ 1 ?", X_train_scaled.std(axis=0).round(3))

### Exercise 3 — Solve with the Normal Equation

Implement the normal-equation solution. Remember to add a bias column of ones to `X_train_scaled`.

In [ ]:
# 1) Add a bias column of ones to X_train_scaled and X_test_scaled
X_train_b = None   # ← TODO: np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_b  = None   # ← TODO: same for X_test_scaled

# 2) Compute theta using the normal equation: theta = (X^T X)^{-1} X^T y
theta = None       # ← TODO: np.linalg.inv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train.values

# 3) Predict on the test set and compute RMSE
y_pred = None      # ← TODO: X_test_b @ theta
rmse = None        # ← TODO: np.sqrt(sklearn_mse(y_test, y_pred))

print(f"Normal Equation test RMSE: {rmse:.4f}")

### Exercise 4 — Solve with QR Decomposition

Use `np.linalg.qr` to factorise `X_train_b`, then solve with `np.linalg.solve`. Compare the resulting theta to the normal equation — they should match.

In [ ]:
# 1) QR decomposition of X_train_b (the matrix WITH the bias column)
Q, R = None, None   # ← TODO: np.linalg.qr(X_train_b)

# 2) Solve R @ theta_qr = Q^T @ y_train
theta_qr = None     # ← TODO: np.linalg.solve(R, Q.T @ y_train.values)

# 3) Check that both solutions match
print("Theta (Normal Eq):", theta)
print("Theta (QR)       :", theta_qr)
print("Are they close?  :", np.allclose(theta, theta_qr))

### **Contribution: Sattam Altwaim** :)